In [0]:
# Verificación de entorno
import sys
print("Versión Pyhton:",sys.version)
print("Versión Spark:",spark.version)

In [0]:
# Definición del esquema - Violencia intrafamiliar 
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

esquema_violencia = StructType([
    StructField("DEPARTAMENTO",  StringType(),  False),
    StructField("MUNICIPIO",     StringType(),  False),
    StructField("CODIGO_DANE",   StringType(),  False),
    StructField("ARMAS_MEDIOS",  StringType(),  False),
    StructField("FECHA_HECHO",   StringType(),  False),
    StructField("GENERO",        StringType(),  False),
    StructField("GRUPO_ETARIO",  StringType(),  False),
    StructField("CANTIDAD",      IntegerType(), False),
])

print("Esquema definido correctamente:")

In [0]:
# Leemos el csv con Spark
df = spark.read \
    .option("header", True) \
    .option("sep", ",") \
    .option("inferSchema", "false") \
    .option("encoding", "UTF-8") \
    .schema(esquema_violencia) \
    .csv("/Volumes/workspace/default/data/Reporte_Delito_Violencia_Intrafamiliar_Polic_a_Nacional.csv")

print("Total filas:    ", df.count())
print("Total columnas: ", len(df.columns))
print("Columnas:       ", df.columns)
df.show(5, truncate=False)

In [0]:
# Crear tabla permanente
spark.sql("DROP TABLE IF EXISTS violencia_intrafamiliar")

df.write.mode("overwrite").saveAsTable("violencia_intrafamiliar")
print("Tabla 'violencia_intrafamiliar' creada correctamente")
print("Registros:", df.count())

In [0]:
# Validaciones con Spark
# Esquema
print("===== ESQUEMA =====")
df.printSchema()

# Descripción estadística
print("\n===== DESCRIPCIÓN ESTADÍSTICA =====")
df.describe("CANTIDAD").show()

# Valores nulos por columna
from pyspark.sql.functions import col, when, sum as spark_sum

print("\n===== VALORES NULOS POR COLUMNA =====")
df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).show()


In [0]:
%sql
-- Metadatos en SQL
DESCRIBE TABLE violencia_intrafamiliar;

In [0]:
%sql
SHOW CREATE TABLE violencia_intrafamiliar;

In [0]:
# En Spark — casos en Antioquia
print("===== SELECT CON FILTRO (PySpark) =====")
df.filter(df.DEPARTAMENTO == "ANTIOQUIA") \
  .select("DEPARTAMENTO", "MUNICIPIO", "FECHA_HECHO", "GENERO", "CANTIDAD") \
  .orderBy("FECHA_HECHO") \
  .show(10, truncate=False)

In [0]:
%sql
-- Equivalente exacto en SQL
SELECT DEPARTAMENTO, MUNICIPIO, FECHA_HECHO, GENERO, CANTIDAD
FROM   violencia_intrafamiliar
WHERE  DEPARTAMENTO = 'ANTIOQUIA'
ORDER BY FECHA_HECHO
LIMIT 10;

In [0]:
# En Spark — casos por departamento
from pyspark.sql.functions import count, sum as spark_sum, col

print("===== GROUP BY DEPARTAMENTO (PySpark) =====")
df.groupBy("DEPARTAMENTO") \
  .agg(
      count("*").alias("registros"),
      spark_sum("CANTIDAD").alias("total_casos")
  ) \
  .orderBy(col("total_casos").desc()) \
  .show(10, truncate=False)

In [0]:
%sql
-- Equivalente en SQL
SELECT   DEPARTAMENTO,
         COUNT(*)        AS registros,
         SUM(CANTIDAD)   AS total_casos
FROM     violencia_intrafamiliar
GROUP BY DEPARTAMENTO
ORDER BY total_casos DESC
LIMIT 10;

In [0]:
%sql
-- GROUP BY por género y grupo etario
SELECT   GENERO,
         GRUPO_ETARIO,
         COUNT(*)      AS registros,
         SUM(CANTIDAD) AS total_casos
FROM     violencia_intrafamiliar
GROUP BY GENERO, GRUPO_ETARIO
ORDER BY total_casos DESC;

# Ventajas y Desventajas: SQL vs Spark

### SQL 
SQL es un lenguaje declarativo conocido por la mayoría de profesionales de datos. Su principal ventaja es la facilidad de uso: permite filtrar, agrupar y agregar datos de forma intuitiva y legible sin necesidad de saber programar. Se integra directamente con herramientas de BI como Power BI o Tableau, lo que facilita llevar resultados a dashboards rápidamente. Su limitación aparece cuando los procesos se vuelven complejos: implementar lógica condicional avanzada, transformaciones iterativas o funciones personalizadas resulta difícil y poco práctico en SQL puro.

### Spark 
PySpark combina el poder del procesamiento distribuido de Spark con todo el ecosistema de Python, permitiendo usar librerías como pandas, scikit-learn o TensorFlow dentro del mismo flujo de trabajo. Es ideal para pipelines complejos, transformaciones personalizadas y proyectos de Machine Learning, y escala sin problema desde miles hasta miles de millones de registros. Su desventaja es la curva de aprendizaje: entender el modelo distribuido de Spark y escribir código eficiente requiere más tiempo y experiencia que escribir una consulta SQL.

**Conclusión:** Para consultas y reportes, mejor SQL.  
Para transformaciones complejas, mejor Spark.  
En Databricks ambas se complementan perfectamente.